In [1]:

import pandas as pd
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================

INPUT_FILE = Path(
    "output/second_reports/tanggal_enriched_cleaned.xlsx"
)

OUTPUT_DIR = Path("output/second_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "attendance_feature_engineered.xlsx"


# ==========================================
# 2. LOAD DATA
# ==========================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

df = pd.read_excel(INPUT_FILE)

print("Dataset loaded successfully!")
print("Total records:", len(df))


# ==========================================
# 3. VALIDATE REQUIRED COLUMNS
# ==========================================

required_columns = [
    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be",
    "location_name",
    "cost_center_code"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ==========================================
# 4. CREATE INDIVIDUAL CLOCK LABELS
# ==========================================

clock_columns = {
    "clock_in_time": "clock_in_time_label",
    "clock_out_time": "clock_out_time_label",
    "clock_in_time_be": "clock_in_time_be_label",
    "clock_out_time_be": "clock_out_time_be_label"
}


def check_clock_value(series):

    value = series.astype("string").str.strip()

    invalid_values = [
        "",
        "NULL",
        "NONE",
        "NAN",
        "NAT",
        "-",
        "<NA>"
    ]

    has_value = (
        series.notna()
        & value.ne("").fillna(False)
        & ~value.str.upper().isin(
            invalid_values
        ).fillna(False)
    )

    return has_value.map({
        True: "Available",
        False: "Missing"
    })


# Generate individual clock labels

for original_col, new_col in clock_columns.items():

    df[new_col] = check_clock_value(
        df[original_col]
    )


# ==========================================
# 5. CREATE COMBINED CLOCK LABELS
# ==========================================

# Clock-in is available if original OR BE exists

has_clock_in_value = (
    df["clock_in_time_label"].eq("Available")
    |
    df["clock_in_time_be_label"].eq("Available")
)

df["clock_in_label"] = has_clock_in_value.map({
    True: "Available",
    False: "Missing"
})


# Clock-out is available if original OR BE exists

has_clock_out_value = (
    df["clock_out_time_label"].eq("Available")
    |
    df["clock_out_time_be_label"].eq("Available")
)

df["clock_out_label"] = has_clock_out_value.map({
    True: "Available",
    False: "Missing"
})

# ==========================================
# 6. CREATE ATTENDANCE FLAGS
# ==========================================

# 1. HAS CLOCK IN
# Available if clock_in_time OR clock_in_time_be exists

df["has_clock_in"] = (
    df["clock_in_time_label"].eq("Available")
    |
    df["clock_in_time_be_label"].eq("Available")
)


# 2. HAS CLOCK OUT
# Available if clock_out_time OR clock_out_time_be exists

df["has_clock_out"] = (
    df["clock_out_time_label"].eq("Available")
    |
    df["clock_out_time_be_label"].eq("Available")
)


# 3. HAS ATTENDANCE ACTIVITY
# Available if attendance_date exists

attendance_date_value = (
    df["attendance_date"]
    .astype("string")
    .str.strip()
)

invalid_date_values = [
    "",
    "NULL",
    "NONE",
    "NAN",
    "NAT",
    "-",
    "<NA>"
]

df["has_attendance_activity"] = (
    df["attendance_date"].notna()
    &
    attendance_date_value.ne("").fillna(False)
    &
    ~attendance_date_value.str.upper()
        .isin(invalid_date_values)
        .fillna(False)
)


print("\nATTENDANCE FLAGS CREATED")

for col in [
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))
    
# ==========================================
# 6. CREATE RECORD COMPLETENESS
# ==========================================

# Complete:
# Clock-in AND clock-out available

# Incomplete:
# Only one available

# Missing:
# Neither available

df["record_completeness"] = "Missing"

df.loc[
    has_clock_in_value | has_clock_out_value,
    "record_completeness"
] = "Incomplete"

df.loc[
    has_clock_in_value & has_clock_out_value,
    "record_completeness"
] = "Complete"


# ==========================================
# 7. CREATE STORE TYPE
# ==========================================

location = (
    df["location_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Classification rules
# Priority follows the order below.

store_conditions = [

    location.str.contains(
        "HYFRESH",
        regex=False,
        na=False
    ),

    location.str.contains(
        "HEAD OFFICE",
        regex=False,
        na=False
    ),

    location.str.contains(
        "DISTRIBUTION CENTER",
        regex=False,
        na=False
    ),

    location.str.contains(
        "HYPERMART",
        regex=False,
        na=False
    ),

    location.str.contains(
        r"\bDC\b",
        regex=True,
        na=False
    ),

    location.str.contains(
        "FMX",
        regex=False,
        na=False
    ),

    location.str.contains(
        "SSO",
        regex=False,
        na=False
    ),

    location.str.contains(
        "BOSTON",
        regex=False,
        na=False
    ),

    location.str.contains(
        "B&B",
        regex=False,
        na=False
    ),

    location.str.contains(
        "TOWAKI",
        regex=False,
        na=False
    ),

    location.str.contains(
        "FOODMART",
        regex=False,
        na=False
    )
]

store_choices = [
    "HFR",
    "HO",
    "DC",
    "HPRM",
    "DC",
    "FMX",
    "SSO",
    "B&B",
    "B&B",
    "TT",
    "FDM"
]

# Initialize store_type

df["store_type"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)

# Assign classifications

for condition, choice in zip(
    store_conditions,
    store_choices
):

    df.loc[
        condition & df["store_type"].isna(),
        "store_type"
    ] = choice

# Unclassified locations

df["store_type"] = df["store_type"].fillna("OTHER")


# ==========================================
# 8. COST CENTER CLASSIFICATION
# ==========================================

# IMPORTANT BUSINESS RULE:
#
# Cost Center starting with 9:
# - Fixed 5-day work week
# - National holidays reduce weekly quota
# - Schedule is FIXED
#
# Other valid Cost Centers:
# - 6-day weekly quota
# - Actual working days are UNKNOWN
# - National holidays do not reduce quota
# - Schedule is UNKNOWN


# Standardize cost center for classification
# Original cost_center_code is preserved.

cost_center = (
    df["cost_center_code"]
    .astype("string")
    .str.strip()
)

# Identify valid cost center codes

invalid_cost_center_values = [
    "",
    "NULL",
    "NONE",
    "NAN",
    "NAT",
    "-",
    "<NA>"
]

valid_cost_center = (
    cost_center.notna()
    &
    cost_center.ne("").fillna(False)
    &
    ~cost_center.str.upper().isin(
        invalid_cost_center_values
    ).fillna(False)
)

# Identify Cost Center 9

is_fixed_5 = (
    valid_cost_center
    &
    cost_center.str.startswith(
        "9",
        na=False
    )
)

is_non_fixed_6 = (
    valid_cost_center
    &
    ~is_fixed_5
)


# ------------------------------------------
# 8A. COST CENTER TYPE
# ------------------------------------------

df["cost_center_type"] = pd.Series(
    "UNKNOWN",
    index=df.index,
    dtype="string"
)

df.loc[
    is_fixed_5,
    "cost_center_type"
] = "FIXED_5"

df.loc[
    is_non_fixed_6,
    "cost_center_type"
] = "NON_FIXED_6"


# ------------------------------------------
# 8B. WEEKLY WORK QUOTA
# ------------------------------------------

df["weekly_work_quota"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int64"
)

df.loc[
    is_fixed_5,
    "weekly_work_quota"
] = 5

df.loc[
    is_non_fixed_6,
    "weekly_work_quota"
] = 6


# ------------------------------------------
# 8C. SCHEDULE TYPE
# ------------------------------------------

df["schedule_type"] = "UNKNOWN"

df.loc[
    is_fixed_5,
    "schedule_type"
] = "FIXED"


# ==========================================
# 9. PRESERVE EXISTING ATTENDANCE FLAGS
# ==========================================

# These columns already exist in the dataset.
# Do NOT recreate or overwrite them.

existing_attendance_flags = [
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]

missing_flags = [
    col for col in existing_attendance_flags
    if col not in df.columns
]

if missing_flags:

    print(
        "\nWARNING: Existing attendance flags "
        "not found:",
        missing_flags
    )

else:

    print(
        "\nExisting attendance flags preserved:"
    )

    print(existing_attendance_flags)


# ==========================================
# 10. EXPORT FEATURE-ENGINEERED DATA
# ==========================================

df.to_excel(
    OUTPUT_FILE,
    index=False
)


# ==========================================
# 11. SUMMARY REPORT
# ==========================================

print("\n==========================================")
print("PROCESS 8 COMPLETED")
print("ATTENDANCE FEATURE ENGINEERING")
print("==========================================")


print("\nTOTAL RECORDS")
print(len(df))


# ------------------------------------------
# Store type distribution
# ------------------------------------------

print("\nSTORE TYPE DISTRIBUTION")

print(
    df["store_type"].value_counts(
        dropna=False
    )
)


# ------------------------------------------
# Individual clock labels
# ------------------------------------------

print("\nINDIVIDUAL CLOCK LABELS")

for col in clock_columns.values():

    print(f"\n{col}")

    print(
        df[col].value_counts(
            dropna=False
        )
    )


# ------------------------------------------
# Combined clock labels
# ------------------------------------------

print("\nCOMBINED CLOCK LABELS")

for col in [
    "clock_in_label",
    "clock_out_label"
]:

    print(f"\n{col}")

    print(
        df[col].value_counts(
            dropna=False
        )
    )


# ------------------------------------------
# Record completeness
# ------------------------------------------

print("\nRECORD COMPLETENESS")

print(
    df["record_completeness"].value_counts(
        dropna=False
    )
)


# ------------------------------------------
# Cost center distribution
# ------------------------------------------

print("\nCOST CENTER TYPE DISTRIBUTION")

print(
    df["cost_center_type"].value_counts(
        dropna=False
    )
)


# ------------------------------------------
# Weekly work quota distribution
# ------------------------------------------

print("\nWEEKLY WORK QUOTA DISTRIBUTION")

print(
    df["weekly_work_quota"].value_counts(
        dropna=False
    ).sort_index()
)


# ------------------------------------------
# Schedule type distribution
# ------------------------------------------

print("\nSCHEDULE TYPE DISTRIBUTION")

print(
    df["schedule_type"].value_counts(
        dropna=False
    )
)


# ------------------------------------------
# Check unknown cost center
# ------------------------------------------

print("\nUNKNOWN COST CENTER RECORDS")

print(
    (df["cost_center_type"] == "UNKNOWN").sum()
)


# ------------------------------------------
# New columns
# ------------------------------------------

print("\nNEW COLUMNS CREATED")

new_columns = [
    "clock_in_time_label",
    "clock_out_time_label",
    "clock_in_time_be_label",
    "clock_out_time_be_label",
    "clock_in_label",
    "clock_out_label",
    "record_completeness",
    "store_type",
    "cost_center_type",
    "weekly_work_quota",
    "schedule_type"
]

print(new_columns)


# ------------------------------------------
# Output
# ------------------------------------------

print("\nOUTPUT FILE")

print(OUTPUT_FILE)

print("\n==========================================")
print("PROCESS 8 SUCCESSFULLY FINISHED")
print("==========================================")


Dataset loaded successfully!
Total records: 27711

ATTENDANCE FLAGS CREATED

has_clock_in
has_clock_in
True     26975
False      736
Name: count, dtype: int64

has_clock_out
has_clock_out
True     23852
False     3859
Name: count, dtype: int64

has_attendance_activity
has_attendance_activity
True    27711
Name: count, dtype: Int64

Existing attendance flags preserved:
['has_clock_in', 'has_clock_out', 'has_attendance_activity']

PROCESS 8 COMPLETED
ATTENDANCE FEATURE ENGINEERING

TOTAL RECORDS
27711

STORE TYPE DISTRIBUTION
store_type
HPRM    19493
DC       2419
FDM      2173
HO       1665
HFR      1345
B&B       472
FMX       113
SSO        25
TT          6
Name: count, dtype: Int64

INDIVIDUAL CLOCK LABELS

clock_in_time_label
clock_in_time_label
Available    26975
Missing        736
Name: count, dtype: int64

clock_out_time_label
clock_out_time_label
Available    23852
Missing       3859
Name: count, dtype: int64

clock_in_time_be_label
clock_in_time_be_label
Available    26562
Miss